In [1]:
import cv2
import numpy as np
from scipy.interpolate import UnivariateSpline

IMAGE_PATH      = "DewarpBook.jpg"
OUTPUT_PATH     = "transformed_image.jpg"
WINDOW_W        = 500    # Width of the interactive window
WINDOW_H        = 700    # Height of the interactive window
NUM_POINTS      = 35     # Number of points to detect on each edge
GRID_ROWS       = 30     # Vertical subdivisions for piecewise warp
PAGE_MARGIN_X   = 50     # Pixels to ignore on left/right edges
PAGE_MARGIN_Y   = 20     # Pixels to ignore on top/bottom edges
ADAPTIVE_WINSZ  = 55     # Block size for adaptive threshold

# Preprocessing: CLAHE + adaptative threshold + morphological ops 
# to create a binary mask of the page
# Enhance contrast with CLAHE
# Adaptive threshold to separate page from background
# Morphological closing/opening to remove noise
def preprocess_mask(gray):
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
    enh   = clahe.apply(gray)
    mask  = cv2.adaptiveThreshold(
        enh, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        ADAPTIVE_WINSZ, 10
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=1)
    # remove borders
    mask[:PAGE_MARGIN_Y, :] = 0
    mask[-PAGE_MARGIN_Y:, :] = 0
    mask[:, :PAGE_MARGIN_X] = 0
    mask[:, -PAGE_MARGIN_X:] = 0
    return mask

#  Blurring to estimate background shading
#  Dividing original gray by that background
#  Normalize result to [0, 255] range
def shading_correction(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bg   = cv2.GaussianBlur(gray, (101,101), 0)
    corr = cv2.divide(gray, bg, scale=255)
    norm = cv2.normalize(corr, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return norm
# trim blank borders by making a mask of pixels if non-blank pixels > 5
# find the bounding rectangle of that mask and crop it
def final_crop(img):
    mask_nz = (img > 5).astype(np.uint8)
    pts     = cv2.findNonZero(mask_nz)
    x,y,w,h = cv2.boundingRect(pts)
    return img[y:y+h, x:x+w]

def label_point(img, x, y, idx, color):
    cv2.circle(img, (x, y), 5, color, -1)
    cv2.putText(img, str(idx+1), (x+5, y-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

# Auto-detect edge points using Canny edge detection
# and smooth results with UnivariateSpline
# y0, y1 = row range to scan, n = number of points to sample

def auto_detect_edge_points(gray, y0, y1, n, axis='top', crop_l=0.2, crop_r=0.98):
    h, w = gray.shape
    xs   = np.linspace(int(crop_l*w), int(crop_r*w), n).astype(int)
    raw  = []
    if axis == 'top':
        edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), 50, 150)
        for x in xs:
            col = edges[y0:y1, x]
            pts = np.where(col > 0)[0]
            raw.append(pts[0]+y0 if len(pts)>0 else (y0+y1)//2)
    else:
        clahe = cv2.createCLAHE(clipLimit=5.0, tileGridSize=(8,8))
        edges = cv2.Canny(clahe.apply(gray), 30, 100)
        for x in xs:
            col = edges[y0:y1, x]
            pts = np.where(col>0)[0]
            raw.append(pts[0]+y0 if len(pts)>0 else (y0+y1)//2)
    # Smooth the raw y-values
    spline = UnivariateSpline(xs, raw, k=3, s=len(xs)*5)
    ys     = spline(xs).astype(int)
    return list(zip(xs, ys))

# Find the most active horizontla band in the lower half of the image
# using canny edge detection and summing edge counts per row
# starting at min_ratio*height. Locate peak and return a band of ±band_h/2 around it.
# y0, y1 = row range to scan, min_ratio = start at this ratio of height
def find_bottom_band(gray, min_ratio=0.5, band_h=40):
    h     = gray.shape[0]
    start = int(h * min_ratio)
    edges = cv2.Canny(gray, 50, 150)
    sums  = np.sum(edges[start:], axis=1)
    peak  = np.argmax(sums) + start
    y0    = max(start, peak - band_h//2)
    y1    = min(h,    peak + band_h//2)
    return y0, y1

def draw_lines(img, pts, zoom, scroll):
    for i in range(NUM_POINTS):
        t = pts[i]
        b = pts[i + NUM_POINTS]
        p1 = (int(t[0]*zoom - scroll[0]), int(t[1]*zoom - scroll[1]))
        p2 = (int(b[0]*zoom - scroll[0]), int(b[1]*zoom - scroll[1]))
        cv2.line(img, p1, p2, (0,0,0), 1)       # black line
        label_point(img, *p1, i, (0,0,255))     # red top
        label_point(img, *p2, i, (255,0,0))     # blue bottom


src = cv2.imread(IMAGE_PATH)
h0, w0 = src.shape[:2]
gray   = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

# Preprocess to get a clean mask of the page
mask = preprocess_mask(gray)
cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
# Pick the largest contour
cnt = max(cnts, key=lambda c: cv2.contourArea(c))

# Approximate page contour by 4 corners
peri  = cv2.arcLength(cnt, True)
approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
if len(approx) == 4:
    quad = approx.reshape(4, 2)
else:
    x,y,w,h = cv2.boundingRect(cnt)
    quad = np.array([[x,y],[x+w,y],[x+w,y+h],[x,y+h]], dtype=np.int32)

# Sort corners TL, TR, BR, BL (not strictly needed for homography)
pts_sorted     = quad[np.argsort(quad[:,1])]
top_sorted     = pts_sorted[:2][np.argsort(pts_sorted[:2,0])]
bot_sorted     = pts_sorted[2:][np.argsort(pts_sorted[2:,0])]
tl, tr, br, bl = map(tuple, [top_sorted[0], top_sorted[1], bot_sorted[1], bot_sorted[0]])

# Initial edge point detection on a downscaled preview
zoom    = WINDOW_H / h0
preview = cv2.resize(src, (int(w0*zoom), int(h0*zoom)))
gp      = cv2.cvtColor(preview, cv2.COLOR_BGR2GRAY)
y0t, y1t = 0, gp.shape[0]//2
tp       = auto_detect_edge_points(gp, y0t, y1t, NUM_POINTS, 'top', 0.2, 0.8)
y0b, y1b = find_bottom_band(gp, 0.5, 40)
bp       = auto_detect_edge_points(gp, y0b, y1b, NUM_POINTS, 'bottom', 0.1, 0.98)
# Scale detected points back to original image size
adjust_pts  = [(int(x/zoom), int(y/zoom)) for x,y in tp + bp]
initial_pts = adjust_pts.copy()

# Interactive adjustment:
#    Left click = drag point
#    Right click + drag = pan image
#    Scroll = zoom
#    r = reset points
#    ENTER = confirm and proceed
selected    = None
pan_active  = False
prev_mouse  = (0,0)
scroll      = [max(0, (int(w0*zoom) - WINDOW_W)//2), 0]

def mouse_cb(event, x, y, flags, param):
    global selected, pan_active, prev_mouse, scroll, zoom
    ox = int((x + scroll[0]) / zoom)
    oy = int((y + scroll[1]) / zoom)
    if event == cv2.EVENT_LBUTTONDOWN:
        # select a point to drag
        for i, pt in enumerate(adjust_pts):
            if abs(pt[0] - ox) < 10 and abs(pt[1] - oy) < 10:
                selected = i
                return
    elif event == cv2.EVENT_RBUTTONDOWN:
        pan_active = True
        prev_mouse = (x, y)
    elif event == cv2.EVENT_MOUSEMOVE:
        if selected is not None:
            adjust_pts[selected] = (ox, oy)
        elif pan_active:
            dx, dy = x - prev_mouse[0], y - prev_mouse[1]
            scroll[0] += dx; scroll[1] += dy
            prev_mouse = (x, y)
    elif event in (cv2.EVENT_LBUTTONUP, cv2.EVENT_RBUTTONUP):
        if event == cv2.EVENT_LBUTTONUP:
            selected = None
        else:
            pan_active = False
    elif event == cv2.EVENT_MOUSEWHEEL:
        # zoom around cursor
        factor = 1.1 if flags > 0 else 0.9
        old_z   = zoom
        img_x   = (x + scroll[0]) / old_z
        img_y   = (y + scroll[1]) / old_z
        zoom    = max(0.2, min(5.0, zoom * factor))
        scroll[0] = int(img_x * zoom - x)
        scroll[1] = int(img_y * zoom - y)

cv2.namedWindow("Adjust", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Adjust", WINDOW_W, WINDOW_H)
cv2.setMouseCallback("Adjust", mouse_cb)
print("Left=drag pt | Right=pan | Scroll=zoom | r=reset | ENTER=confirm")

while True:
    vis = cv2.resize(src, None, fx=zoom, fy=zoom)
    H_vis, W_vis = vis.shape[:2]
    scroll[0] = np.clip(scroll[0], 0, W_vis - WINDOW_W)
    scroll[1] = np.clip(scroll[1], 0, H_vis - WINDOW_H)
    win = vis[scroll[1]:scroll[1]+WINDOW_H, scroll[0]:scroll[0]+WINDOW_W].copy()
    draw_lines(win, adjust_pts, zoom, scroll)
    cv2.imshow("Adjust", win)
    k = cv2.waitKey(10) & 0xFF
    if k == ord('r'):
        adjust_pts = initial_pts.copy()
    elif k == 13:  # ENTER
        break

cv2.destroyAllWindows()

# Piecewise homography + stitch:
dst     = np.zeros_like(src)
strip_w = w0 // (NUM_POINTS - 1)
cell_h  = h0 // GRID_ROWS
for i in range(NUM_POINTS - 1):
    t0, t1 = adjust_pts[i], adjust_pts[i+1]
    b0, b1 = adjust_pts[i+NUM_POINTS], adjust_pts[i+NUM_POINTS+1]
    for j in range(GRID_ROWS):
        a0, a1 = j/GRID_ROWS, (j+1)/GRID_ROWS
        # build source and dest quads
        srcQ = np.float32([
            [(1-a0)*t0[0] + a0*b0[0], (1-a0)*t0[1] + a0*b0[1]],
            [(1-a0)*t1[0] + a0*b1[0], (1-a0)*t1[1] + a0*b1[1]],
            [(1-a1)*t1[0] + a1*b1[0], (1-a1)*t1[1] + a1*b1[1]],
            [(1-a1)*t0[0] + a1*b0[0], (1-a1)*t0[1] + a1*b0[1]],
        ])
        dstQ = np.float32([
            [i*strip_w,     j*cell_h],
            [(i+1)*strip_w, j*cell_h],
            [(i+1)*strip_w, (j+1)*cell_h],
            [i*strip_w,     (j+1)*cell_h],
        ])
        H    = cv2.getPerspectiveTransform(srcQ, dstQ)
        wimg = cv2.warpPerspective(src, H, (w0, h0))
        # composite only this cell
        m   = np.zeros((h0, w0), dtype=np.uint8)
        cv2.fillConvexPoly(m, dstQ.astype(int), 255)
        dst = np.where(m[..., None]==255, wimg, dst)

# Postprocess: shading correction + final crop
shaded = shading_correction(dst)
final  = final_crop(shaded)

cv2.imwrite(OUTPUT_PATH, final)


Left=drag pt | Right=pan | Scroll=zoom | r=reset | ENTER=confirm


KeyboardInterrupt: 